# Download ImageNet Validation Set to Google Drive
Run this once. Downloads from HuggingFace and uploads to your Drive folder without mounting all of Drive.

Prerequisites:
1. Accept dataset terms at huggingface.co/datasets/imagenet-1k
2. Create a read token at huggingface.co/settings/tokens

In [ ]:
!pip install -q datasets huggingface_hub google-api-python-client google-auth-httplib2

In [ ]:
from huggingface_hub import login
login()  # paste your HF read token when prompted

In [ ]:
from datasets import load_dataset, Dataset

ds_stream = load_dataset('ILSVRC/imagenet-1k', split='validation', streaming=True)

def gen():
    for example in ds_stream:
        yield example

ds = Dataset.from_generator(gen, features=ds_stream.features)
ds.save_to_disk('/content/imagenet_val')
print(f'Saved {len(ds)} images to /content/imagenet_val')

In [ ]:
# Zip the dataset folder before uploading
!cd /content && zip -qr imagenet_val.zip imagenet_val/
!du -sh /content/imagenet_val.zip

In [ ]:
# Upload zip to your specific Drive folder (no full Drive mount)
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

FOLDER_ID = '16axcxIfAwW0ebpc_0NuJSeSy_RDY_zqt'

creds, _ = google.auth.default()
service = build('drive', 'v3', credentials=creds)

meta  = {'name': 'imagenet_val.zip', 'parents': [FOLDER_ID]}
media = MediaFileUpload('/content/imagenet_val.zip', resumable=True)
f = service.files().create(body=meta, media_body=media, fields='id').execute()
print('Uploaded, file id:', f['id'])